In [ ]:
import pandas as pd
from xGATE.utilities import *
import random
import warnings
import pip
import spicy
from scipy import io

warnings.filterwarnings('ignore')
random.seed(12)
np.random.seed(12)
torch.manual_seed(12)
    
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

matrix = io.mmread("/work/of21_work/hepatocyte_pre_processed_coexp.mtx")

# Convert the sparse matrix to a dense matrix
dense_matrix = matrix.toarray()

# Create a Pandas DataFrame from the dense matrix
df = pd.DataFrame(dense_matrix)

# Read the CSV file without column names
row_names = pd.read_csv('/work/of21_work/rownames_hepatocyte_pre_processed', header=None)
row_names_list = row_names.iloc[:, 0].tolist()
df.index = row_names_list
df.columns = row_names_list

adj_matrix = df
adj_matrix.index = df.index

adj_matrix.columns = df.index

adj_matrix = convert_gene_ids(adj_matrix, "ensembl")

G_s = create_network_from_adj_matrix(adj_matrix)

categorized_pathways = get_categorized_pathways()

41 input query terms found no hit:	['ENSG00000240401', 'ENSG00000212978', 'ENSG00000239467', 'ENSG00000223797', 'ENSG00000269028', 'ENS
41 input query terms found no hit:	['ENSG00000240401', 'ENSG00000212978', 'ENSG00000239467', 'ENSG00000223797', 'ENSG00000269028', 'ENS


In [2]:
test_pathways = [
  "Carbon metabolism", "Biosynthesis of amino acids", "Biosynthesis of cofactors",
  "Glycolysis / Gluconeogenesis", "Pentose and glucuronate interconversions",
  "Ascorbate and aldarate metabolism", "Pyruvate metabolism", "Fatty acid degradation",
  "Primary bile acid biosynthesis", "Steroid hormone biosynthesis",
  "Arachidonic acid metabolism", "Linoleic acid metabolism",
  "Biosynthesis of unsaturated fatty acids", "Glycine, serine and threonine metabolism",
  "Cysteine and methionine metabolism", "Tyrosine metabolism",
  "Taurine and hypotaurine metabolism", "Retinol metabolism",
  "Porphyrin metabolism", "Metabolism of xenobiotics by cytochrome P450",
  "Cholesterol metabolism", "Caffeine metabolism", "Drug metabolism", "Thyroid cancer",
    "Shigellosis", 
    "Colorectal cancer",
    "Pancreatic cancer",
    "Hepatocellular carcinoma",
    "Gastric cancer",
    "Glioma",
    "Thyroid cancer",
    "Acute myeloid leukemia",
    "Chronic myeloid leukemia",
    "Basal cell carcinoma",
    "Melanoma",
    "Renal cell carcinoma",
    "Bladder cancer",
    "Prostate cancer",
    "Endometrial cancer",
    "Breast cancer",
    "Small cell lung cancer",
    "Non-small cell lung cancer"
]

pathway_results = analyze_pathways(G_s, test_pathways, categorized_pathways ,num_walks=200, max_walk_length = 200, null_dist_size = 200)

Pathway: Carbon metabolism
p-value: 0.004975124378109453
Z-Score: 29.89280204581356

Pathway: Biosynthesis of amino acids
p-value: 0.004975124378109453
Z-Score: 120.30239055914437

Pathway: Biosynthesis of cofactors
p-value: 0.1044776119402985
Z-Score: 0.029755613105754388

Pathway: Glycolysis / Gluconeogenesis
p-value: 0.004975124378109453
Z-Score: 939.5807910653701

Pathway: Pentose and glucuronate interconversions
p-value: 0.004975124378109453
Z-Score: 97.05159725492213

Pathway: Ascorbate and aldarate metabolism
p-value: 0.004975124378109453
Z-Score: 36.11420807782295

Pathway: Pyruvate metabolism
p-value: 0.004975124378109453
Z-Score: 42.55844971655226

Pathway: Fatty acid degradation
p-value: 0.004975124378109453
Z-Score: 112.81209094257312

Pathway: Primary bile acid biosynthesis
p-value: 0.4975124378109453
Z-Score: -0.12523041943328356

Pathway: Steroid hormone biosynthesis
p-value: 0.004975124378109453
Z-Score: 318.7162516666247

Pathway: Arachidonic acid metabolism
p-value: 0

In [3]:
import pandas as pd
import statsmodels.stats.multitest as smm

# Extract original p-values
pvals = pathway_results['p-value'].values

# Apply the Benjamini-Hochberg FDR correction
reject, pvals_corrected, alphacSidak, alphacBonf = smm.multipletests(pvals, alpha=0.05, method='fdr_bh')

# Create a new column in the DataFrame with the corrected p-values
pathway_results['p-value_corrected'] = pvals_corrected


In [4]:
pathway_results

,pathway,p-value,z-score,p-value_corrected
0,Carbon metabolism,0.004975,29.892802,0.010998
1,Biosynthesis of amino acids,0.004975,120.302391,0.010998
2,Biosynthesis of cofactors,0.104478,0.029756,0.175522
3,Glycolysis / Gluconeogenesis,0.004975,939.580791,0.010998
4,Pentose and glucuronate interconversions,0.004975,97.051597,0.010998
5,Ascorbate and aldarate metabolism,0.004975,36.114208,0.010998
6,Pyruvate metabolism,0.004975,42.558450,0.010998
7,Fatty acid degradation,0.004975,112.812091,0.010998
8,Primary bile acid biosynthesis,0.497512,-0.125230,0.652985
9,Steroid hormone biosynthesis,0.004975,318.716252,0.010998
